In [1]:
import pandas as pd

df = pd.read_csv("AGMARKNET.csv")

df.head()

,state,district,market,commodity,variety,grade,arrival_date,min_price,max_price,modal_price
0,Odisha,Sambalpur,Sahaspur APMC,Little gourd(Kundru),Other,Local,13/07/2026,1600.0,2000.0,1800.0
1,Odisha,Mayurbhanja,Baripada APMC,Ridgeguard(Tori),Ridgeguard(Tori),Local,13/07/2026,4000.0,5000.0,4500.0
2,Keralam,Kozhikode(Calicut),Mukkom Market,Apple,Other,Large,13/07/2026,26000.0,27000.0,26500.0
3,Keralam,Kozhikode(Calicut),Mukkom Market,Bitter gourd,Bitter Gourd,FAQ,13/07/2026,6000.0,6400.0,6200.0
4,Keralam,Kozhikode(Calicut),Mukkom Market,Carrot,Carrot,FAQ,13/07/2026,6000.0,6500.0,6200.0


In [2]:
market_df = (
    df[["state", "district", "market"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

market_df.shape

(1391, 3)

In [3]:
pip install geopy

Note: you may need to restart the kernel to use updated packages.


In [7]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
import time

In [5]:
geolocator = Nominatim(user_agent="agmarknet_project")

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1
)

In [8]:
from geopy.geocoders import ArcGIS
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
import time

# ======================================
# ArcGIS Geocoder
# ======================================

geolocator = ArcGIS(timeout=30)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=0.4,
    max_retries=5,
    error_wait_seconds=5,
    swallow_exceptions=True
)

# ======================================
# Create columns only once
# ======================================

if "latitude" not in market_df.columns:
    market_df["latitude"] = None

if "longitude" not in market_df.columns:
    market_df["longitude"] = None

total = len(market_df)

print("="*60)
print(f"Total Markets : {total}")
print("ArcGIS Geocoding Started...")
print("="*60)

# ======================================
# Start Geocoding
# ======================================

for i, row in market_df.iterrows():

    # Skip already completed rows
    if pd.notna(row["latitude"]):
        continue

    try:

        # First try Market + District + State
        address = f"{row['market']}, {row['district']}, {row['state']}, India"

        location = geocode(address)

        # Fallback
        if location is None:

            address = f"{row['district']}, {row['state']}, India"

            location = geocode(address)

        if location:

            market_df.loc[i, "latitude"] = location.latitude
            market_df.loc[i, "longitude"] = location.longitude

            print(f"✓ {i+1}/{total}  {row['market']}")

        else:

            print(f"✗ {i+1}/{total}  Not Found")

    except Exception as e:

        print(f"⚠ Error at row {i+1}: {e}")

    # Save every 10 records
    if (i + 1) % 10 == 0:

        market_df.to_csv(
            "checkpoint_market_locations.csv",
            index=False
        )

        print("Checkpoint Saved")

# ======================================
# Save Final Dataset
# ======================================

market_df.to_csv(
    "Dataset2_Market_Locations.csv",
    index=False
)

print("\n")
print("="*60)
print("✅ Geocoding Completed Successfully")
print("="*60)

print("\nDataset Shape :", market_df.shape)

print("\nFirst 5 Rows")

display(market_df.head())

Total Markets : 1391
ArcGIS Geocoding Started...
✓ 1/1391  Sahaspur APMC
✓ 2/1391  Baripada APMC
✓ 3/1391  Mukkom Market
✓ 4/1391  Mananthavady Market
✓ 5/1391  Jasdan APMC
✓ 6/1391  Kendupatna(Niali) APMC
✓ 7/1391  Kesinga APMC
✓ 8/1391  Sidhi APMC
✓ 9/1391  Davangere APMC
✓ 10/1391  Kuttoor Market
Checkpoint Saved
✓ 11/1391  Somala APMC
✓ 12/1391  Cherthala Market
✓ 13/1391  Thrippunithura Market
✓ 14/1391  Balasore APMC
✓ 15/1391  Pananchery VFPCK Market
✓ 16/1391  Kollengode Market
✓ 17/1391  Kattappana Market
✓ 18/1391  Himatnagar APMC
✓ 19/1391  Santar Bazaar APMC
✓ 20/1391  Radaur APMC
Checkpoint Saved
✓ 21/1391  Madlauda APMC
✓ 22/1391  Khanina APMC
✓ 23/1391  Kosli APMC
✓ 24/1391  Sampla APMC
✓ 25/1391  Ganaur APMC
✓ 26/1391  Rania(Jiwan nagar) APMC
✓ 27/1391  Shahabad APMC
✓ 28/1391  Akluj APMC
✓ 29/1391  FerozpurZirkha(Nagina) APMC
✓ 30/1391  Kollam Market
Checkpoint Saved
✓ 31/1391  New Grain Market , Panchkula APMC
✓ 32/1391  Anchal Market
✓ 33/1391  Bijaynagar APMC
✓ 34/1

,state,district,market,latitude,longitude
0,Odisha,Sambalpur,Sahaspur APMC,21.341608,83.958399
1,Odisha,Mayurbhanja,Baripada APMC,21.954103,86.742898
2,Keralam,Kozhikode(Calicut),Mukkom Market,11.24802,75.7804
3,Keralam,Wayanad,Mananthavady Market,11.67847,75.975073
4,Gujarat,Rajkot,Jasdan APMC,22.0375,71.206497


In [1]:
import pandas as pd

market_df = pd.read_csv("Dataset2_Market_Locations.csv")

market_df.head()

,state,district,market,latitude,longitude
0,Odisha,Sambalpur,Sahaspur APMC,21.341608,83.958399
1,Odisha,Mayurbhanja,Baripada APMC,21.954103,86.742898
2,Keralam,Kozhikode(Calicut),Mukkom Market,11.248020,75.780400
3,Keralam,Wayanad,Mananthavady Market,11.678470,75.975073
4,Gujarat,Rajkot,Jasdan APMC,22.037500,71.206497


In [3]:
market_df.shape

(1391, 5)

In [4]:
print(market_df.shape)
market_df.head()

(1391, 5)


,state,district,market,latitude,longitude
0,Odisha,Sambalpur,Sahaspur APMC,21.341608,83.958399
1,Odisha,Mayurbhanja,Baripada APMC,21.954103,86.742898
2,Keralam,Kozhikode(Calicut),Mukkom Market,11.248020,75.780400
3,Keralam,Wayanad,Mananthavady Market,11.678470,75.975073
4,Gujarat,Rajkot,Jasdan APMC,22.037500,71.206497


In [5]:
market_df.isnull().sum()

state        0
district     0
market       0
latitude     0
longitude    0
dtype: int64

In [6]:
print("Duplicate Rows :", market_df.duplicated().sum())

Duplicate Rows : 0


In [7]:
print("Duplicate Markets :", market_df["market"].duplicated().sum())

Duplicate Markets : 10


In [8]:
print(
    ((market_df["latitude"] < -90) | (market_df["latitude"] > 90)).sum()
)

print(
    ((market_df["longitude"] < -180) | (market_df["longitude"] > 180)).sum()
)

0
0


In [9]:
market_df["market"].str.strip().eq("").sum()

0

In [10]:
duplicate_markets = market_df[
    market_df["market"].duplicated(keep=False)
].sort_values("market")

duplicate_markets

,state,district,market,latitude,longitude
1368,Madhya Pradesh,Rewa,Baikunthpur APMC,24.726101,81.406303
1256,Chattisgarh,Koria,Baikunthpur APMC,23.262199,82.559998
1252,Madhya Pradesh,Sagar,Banda APMC,24.038601,78.961304
1139,Uttar Pradesh,Banda,Banda APMC,25.480700,80.331902
358,Haryana,Fatehabad,Fatehabad APMC,29.512699,75.451103
372,Uttar Pradesh,Agra,Fatehabad APMC,27.026123,78.302490
267,Madhya Pradesh,Dindori,Gorakhpur APMC,22.729688,80.967901
633,Uttar Pradesh,Gorakhpur,Gorakhpur APMC,26.760760,83.373705
852,Uttar Pradesh,Shahjahanpur,Jalalabad APMC,27.725100,79.653603
606,Punjab,Fazilka,Jalalabad APMC,30.604799,74.250999


In [11]:
duplicate_markets["market"].value_counts()

market
Baikunthpur APMC    2
Banda APMC          2
Fatehabad APMC      2
Gorakhpur APMC      2
Jalalabad APMC      2
Mansa APMC          2
Pratapgarh APMC     2
Sagar APMC          2
Shahabad APMC       2
Sultanpur APMC      2
Name: count, dtype: int64

In [12]:
duplicate_markets[
    ["state", "district", "market"]
]

,state,district,market
1368,Madhya Pradesh,Rewa,Baikunthpur APMC
1256,Chattisgarh,Koria,Baikunthpur APMC
1252,Madhya Pradesh,Sagar,Banda APMC
1139,Uttar Pradesh,Banda,Banda APMC
358,Haryana,Fatehabad,Fatehabad APMC
372,Uttar Pradesh,Agra,Fatehabad APMC
267,Madhya Pradesh,Dindori,Gorakhpur APMC
633,Uttar Pradesh,Gorakhpur,Gorakhpur APMC
852,Uttar Pradesh,Shahjahanpur,Jalalabad APMC
606,Punjab,Fazilka,Jalalabad APMC


In [14]:
market_df.to_csv(
    "Dataset2_Cleaned_Market_Locations.csv",
    index=False
)

print("Dataset 2 Saved Successfully")

Dataset 2 Saved Successfully
